# Project Objective

The objective of this project is to build an intelligent financial risk
assessment platform that uses machine learning to:

1. Predict EMI eligibility as:
   - Eligible
   - High_Risk
   - Not_Eligible

2. Predict the maximum safe monthly EMI amount.

The complete platform will include data preprocessing, exploratory data
analysis, feature engineering, classification and regression models,
MLflow experiment tracking, model selection, a Streamlit application,
CRUD operations, and cloud deployment.

#**Data Loading & Initial Audit**

In [1]:
# Import libraries
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
# Locate and load the dataset

DATA_PATH = Path("../data/emi_prediction_dataset.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("emi_prediction_dataset.csv")
df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

print("Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Dataset loaded successfully.
Rows    : 404,800
Columns : 27


In [3]:
df.head()


,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,family_size,dependents,school_fees,college_fees,travel_expenses,groceries_utilities,other_monthly_expenses,existing_loans,current_emi_amount,credit_score,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi
0,38,Female,Married,Professional,82600.0,Private,0.90,Mid-size,Rented,"20,000.00",3,2,0.00,0.00,"7,200.00","19,500.00","13,200.00",Yes,"23,700.00",660.00,303200.0,"70,200.00",Personal Loan EMI,"850,000.00",15,Not_Eligible,500.00
1,38,Female,Married,Graduate,21500.0,Private,7.00,MNC,Family,0.00,2,1,"5,100.00",0.00,"1,400.00","5,400.00","3,500.00",Yes,"4,100.00",714.00,92500.0,"26,900.00",E-commerce Shopping EMI,"128,000.00",19,Not_Eligible,700.00
2,38,Male,Married,Professional,86100.0,Private,5.80,Startup,Own,0.00,4,3,0.00,0.00,"10,200.00","19,400.00","6,000.00",No,0.00,650.00,672100.0,"324,200.00",Education EMI,"306,000.00",16,Eligible,"27,775.00"
3,58,Female,Married,High School,66800.0,Private,2.20,Mid-size,Own,0.00,5,4,"11,400.00",0.00,"6,200.00","11,900.00","7,900.00",No,0.00,685.00,440900.0,"178,100.00",Vehicle EMI,"304,000.00",83,Eligible,"16,170.00"
4,48,Female,Married,Professional,57300.0,Private,3.40,Mid-size,Family,0.00,4,3,"9,400.00","21,300.00","3,600.00","16,200.00","8,100.00",No,0.00,770.00,97300.0,"28,200.00",Home Appliances EMI,"252,000.00",7,Not_Eligible,500.00


In [4]:
# Basic statistical overview
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,404800,31,38,119883,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,404800,8,Male,237427,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital_status,404800,2,Married,307837,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,402396,4,Graduate,181015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
monthly_salary,404800,13662,18000.0,4159,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employment_type,404800,3,Private,283099,NaN,NaN,NaN,NaN,NaN,NaN,NaN
years_of_employment,"404,800.00",NaN,NaN,NaN,5.36,6.08,0.50,1.20,3.20,7.20,36.00
company_type,404800,5,Large Indian,121139,NaN,NaN,NaN,NaN,NaN,NaN,NaN
house_type,404800,3,Rented,161601,NaN,NaN,NaN,NaN,NaN,NaN,NaN
monthly_rent,"402,374.00",NaN,NaN,NaN,"5,828.45","8,648.60",0.00,0.00,0.00,"10,600.00","80,000.00"


In [5]:
# Data Quality Overview
quality_summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Non-Null": df.notna().sum(),
    "Missing": df.isna().sum(),
    "Missing %": df.isna().mean() * 100,
    "Unique": df.nunique()
})

quality_summary.sort_values("Missing", ascending=False)

,Data Type,Non-Null,Missing,Missing %,Unique
monthly_rent,float64,402374,2426,0.60,4396
bank_balance,object,402374,2426,0.60,12261
credit_score,float64,402380,2420,0.60,427
education,object,402396,2404,0.59,4
emergency_fund,float64,402449,2351,0.58,5486
marital_status,object,404800,0,0.00,2
monthly_salary,object,404800,0,0.00,13662
years_of_employment,float64,404800,0,0.00,356
gender,object,404800,0,0.00,8
employment_type,object,404800,0,0.00,3


In [6]:
# Duplicate Analysis
duplicate_count = df.duplicated().sum()
print(f"Duplicate records: {duplicate_count:,}")
print(f"Duplicate percentage: {(duplicate_count / len(df)) * 100:.2f}%")


Duplicate records: 0
Duplicate percentage: 0.00%


In [7]:
# Categorical Consistency
categorical_columns = [
    "gender",
    "marital_status",
    "education",
    "employment_type",
    "company_type",
    "house_type",
    "existing_loans",
    "emi_scenario",
    "emi_eligibility"
]

categorical_summary = pd.DataFrame({
    "Unique Values": df[categorical_columns].nunique(dropna=True),
    "Missing Values": df[categorical_columns].isna().sum()
})

categorical_summary

,Unique Values,Missing Values
gender,8,0
marital_status,2,0
education,4,2404
employment_type,3,0
company_type,5,0
house_type,3,0
existing_loans,2,0
emi_scenario,5,0
emi_eligibility,3,0


In [8]:
# Numerical Validation
# Columns expected to be numerical according to the project specification
numeric_columns = [
    "age",
    "monthly_salary",
    "years_of_employment",
    "monthly_rent",
    "family_size",
    "dependents",
    "school_fees",
    "college_fees",
    "travel_expenses",
    "groceries_utilities",
    "other_monthly_expenses",
    "current_emi_amount",
    "credit_score",
    "bank_balance",
    "emergency_fund",
    "requested_amount",
    "requested_tenure",
    "max_monthly_emi"
]

# Convert only for inspection â€” DO NOT modify df
numeric_inspection = pd.DataFrame()

for column in numeric_columns:
    numeric_inspection[column] = pd.to_numeric(
        df[column].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

numeric_inspection.describe().T

,count,mean,std,min,25%,50%,75%,max
age,"404,797.00",38.88,9.30,26.00,32.00,38.00,48.00,59.00
monthly_salary,"402,807.00","59,509.34","43,388.30","3,967.00","35,400.00","51,700.00","73,000.00","499,970.00"
years_of_employment,"404,800.00",5.36,6.08,0.50,1.20,3.20,7.20,36.00
monthly_rent,"402,374.00","5,828.45","8,648.60",0.00,0.00,0.00,"10,600.00","80,000.00"
family_size,"404,800.00",2.94,1.08,1.00,2.00,3.00,4.00,5.00
dependents,"404,800.00",1.94,1.08,0.00,1.00,2.00,3.00,4.00
school_fees,"404,800.00","4,624.58","5,061.07",0.00,0.00,"3,000.00","9,000.00","15,000.00"
college_fees,"404,800.00","4,066.25","7,319.34",0.00,0.00,0.00,"6,500.00","25,000.00"
travel_expenses,"404,800.00","5,687.50","3,392.67",600.00,"3,200.00","4,900.00","7,400.00","30,300.00"
groceries_utilities,"404,800.00","12,805.00","6,993.85","1,800.00","7,700.00","11,400.00","16,400.00","71,200.00"


In [9]:
# Range Validation

range_checks = {
    "age": (25, 60),
    "monthly_salary": (15000, 200000),
    "credit_score": (300, 850)
}

range_results = []

for column, (lower, upper) in range_checks.items():
    values = numeric_inspection[column]

    below = (values < lower).sum()
    above = (values > upper).sum()

    range_results.append({
        "Column": column,
        "Expected Min": lower,
        "Expected Max": upper,
        "Below Minimum": below,
        "Above Maximum": above,
        "Out of Range": below + above
    })

range_results = pd.DataFrame(range_results)

range_results

,Column,Expected Min,Expected Max,Below Minimum,Above Maximum,Out of Range
0,age,25,60,0,0,0
1,monthly_salary,15000,200000,6417,3605,10022
2,credit_score,300,850,1566,3210,4776


In [10]:
# Consolidated data-quality findings

malformed_age_count = df["age"].astype(str).str.endswith(".0.0").sum()

data_quality_findings = pd.DataFrame([
    {
        "Issue": "Duplicate records",
        "Finding": f"{df.duplicated().sum():,} duplicate rows",
        "Status": "No issue"
    },
    {
        "Issue": "Missing values",
        "Finding": f"{df.isna().sum().sum():,} missing cells",
        "Status": "Requires handling"
    },
    {
        "Issue": "Age formatting",
        "Finding": f"{malformed_age_count:,} malformed values",
        "Status": "Requires cleaning"
    },
    {
        "Issue": "Gender consistency",
        "Finding": f"{df['gender'].nunique()} representations",
        "Status": "Requires standardization"
    },
    {
        "Issue": "Monthly salary range",
        "Finding": f"{((numeric_inspection['monthly_salary'] < 15000) | (numeric_inspection['monthly_salary'] > 200000)).sum():,} outside documented range",
        "Status": "Requires investigation"
    },
    {
        "Issue": "Credit score range",
        "Finding": f"{((numeric_inspection['credit_score'] < 300) | (numeric_inspection['credit_score'] > 850)).sum():,} outside documented range",
        "Status": "Requires investigation"
    },
    {
        "Issue": "Classification target",
        "Finding": f"{df['emi_eligibility'].nunique()} classes",
        "Status": "Valid"
    },
    {
        "Issue": "Regression target",
        "Finding": f"{df['max_monthly_emi'].isna().sum():,} missing values",
        "Status": "Valid"
    }
])

display(data_quality_findings)

,Issue,Finding,Status
0,Duplicate records,0 duplicate rows,No issue
1,Missing values,"12,027 missing cells",Requires handling
2,Age formatting,3 malformed values,Requires cleaning
3,Gender consistency,8 representations,Requires standardization
4,Monthly salary range,"10,022 outside documented range",Requires investigation
5,Credit score range,"4,776 outside documented range",Requires investigation
6,Classification target,3 classes,Valid
7,Regression target,0 missing values,Valid


In [11]:
# Data Cleaning & Standardization

df_raw = df.copy()
# 1. Numeric column conversion
numeric_columns = [
    "age",
    "monthly_salary",
    "years_of_employment",
    "monthly_rent",
    "family_size",
    "dependents",
    "school_fees",
    "college_fees",
    "travel_expenses",
    "groceries_utilities",
    "other_monthly_expenses",
    "current_emi_amount",
    "credit_score",
    "bank_balance",
    "emergency_fund",
    "requested_amount",
    "requested_tenure",
    "max_monthly_emi"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# 2. Standardize categorical values
# Gender has multiple representations in the raw data.
gender_mapping = {
    "M": "Male",
    "male": "Male",
    "MALE": "Male",
    "F": "Female",
    "female": "Female",
    "FEMALE": "Female",
    "Male": "Male",
    "Female": "Female"
}

df["gender"] = df["gender"].map(gender_mapping)

# Clean surrounding whitespace in categorical columns
categorical_columns = [
    "gender",
    "marital_status",
    "education",
    "employment_type",
    "company_type",
    "house_type",
    "existing_loans",
    "emi_scenario",
    "emi_eligibility"
]

for column in categorical_columns:
    df[column] = df[column].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )


# 3. Missing-value handling
# Numerical variables:
# Median imputation is robust to skewed financial distributions.
for column in numeric_columns:
    if df[column].isna().any():
        df[column] = df[column].fillna(df[column].median())

# Categorical variables:
# Use the most frequent category for missing categorical values.
for column in categorical_columns:
    if df[column].isna().any():
        df[column] = df[column].fillna(df[column].mode()[0])

print("Data cleaning and standardization completed.")

Data cleaning and standardization completed.


In [12]:
# Post-cleaning validation

validation_summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Values": df.isna().sum(),
    "Unique Values": df.nunique()
})

print(f"Shape after cleaning: {df.shape}")
print(f"Remaining missing cells: {df.isna().sum().sum():,}")
print(f"Remaining duplicate rows: {df.duplicated().sum():,}")

display(validation_summary)

Shape after cleaning: (404800, 27)
Remaining missing cells: 0


Remaining duplicate rows: 0


,Data Type,Missing Values,Unique Values
age,float64,0,15
gender,object,0,2
marital_status,object,0,2
education,object,0,4
monthly_salary,float64,0,12780
employment_type,object,0,3
years_of_employment,float64,0,356
company_type,object,0,5
house_type,object,0,3
monthly_rent,float64,0,4396


#**Data Spliting**

In [13]:
from sklearn.model_selection import train_test_split

# Separate features and targets
X = df.drop(columns=["emi_eligibility", "max_monthly_emi"])
y_classification = df["emi_eligibility"]
y_regression = df["max_monthly_emi"]

# First split: 70% training, 30% temporary
X_train, X_temp, y_class_train, y_class_temp, y_reg_train, y_reg_temp = train_test_split(
    X,
    y_classification,
    y_regression,
    test_size=0.30,
    random_state=42,
    stratify=y_classification
)

# Second split: divide the temporary 30% equally
# 15% validation + 15% test
X_val, X_test, y_class_val, y_class_test, y_reg_val, y_reg_test = train_test_split(
    X_temp,
    y_class_temp,
    y_reg_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_class_temp
)

print("Dataset Split")
print("-" * 40)
print(f"Training   : {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation : {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test       : {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")

print("\nClassification Distribution")
print("-" * 40)

split_distribution = pd.DataFrame({
    "Train %": y_class_train.value_counts(normalize=True) * 100,
    "Validation %": y_class_val.value_counts(normalize=True) * 100,
    "Test %": y_class_test.value_counts(normalize=True) * 100
}).round(2)

display(split_distribution)

Dataset Split
----------------------------------------
Training   : 283,360 (70.0%)
Validation : 60,720 (15.0%)
Test       : 60,720 (15.0%)

Classification Distribution
----------------------------------------


,Train %,Validation %,Test %
emi_eligibility,,,
Not_Eligible,77.29,77.29,77.29
Eligible,18.39,18.39,18.39
High_Risk,4.32,4.32,4.32


#**Exploratory Data Analysis (EDA)**

In [14]:
# Target variable analysis

target_summary = pd.DataFrame({
    "Count": df["emi_eligibility"].value_counts(),
    "Percentage": df["emi_eligibility"].value_counts(normalize=True) * 100
}).round(2)

print("EMI Eligibility Distribution")
display(target_summary)

print("\nMaximum Monthly EMI â€” Statistical Summary")
display(df["max_monthly_emi"].describe().to_frame().T)

print("\nMaximum Monthly EMI â€” Skewness")
print(f"{df['max_monthly_emi'].skew():.2f}")

EMI Eligibility Distribution


,Count,Percentage
emi_eligibility,,
Not_Eligible,312868,77.29
Eligible,74444,18.39
High_Risk,17488,4.32



Maximum Monthly EMI â€” Statistical Summary


,count,mean,std,min,25%,50%,75%,max
max_monthly_emi,"404,800.00","6,763.60","7,741.26",500.00,500.00,"4,211.20","9,792.00","91,040.40"



Maximum Monthly EMI â€” Skewness
1.97


###Univariate Analysis

In [15]:
import matplotlib.pyplot as plt

numerical_eda = [
    "monthly_salary",
    "credit_score",
    "requested_amount",
    "max_monthly_emi"
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, column in zip(axes.flat, numerical_eda):
    ax.hist(
        df[column],
        bins=40,
        edgecolor="black",
        linewidth=0.7
    )

    ax.set_title(column.replace("_", " ").title())
    ax.set_xlabel(column.replace("_", " ").title())
    ax.set_ylabel("Frequency")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_20584\263056605.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
categorical_eda = [
    "gender",
    "employment_type",
    "emi_scenario"
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, column in zip(axes, categorical_eda):
    counts = df[column].value_counts()
    counts.plot(kind="bar", ax=ax)
    ax.set_title(column.replace("_", " ").title())
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_20584\2044898139.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


###Bivariate Analysis

In [17]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Salary vs EMI Eligibility
df.boxplot(
    column="monthly_salary",
    by="emi_eligibility",
    ax=axes[0],
    grid=False
)
axes[0].set_title("Monthly Salary vs EMI Eligibility")
axes[0].set_xlabel("EMI Eligibility")
axes[0].set_ylabel("Monthly Salary")

# 2. Credit Score vs EMI Eligibility
df.boxplot(
    column="credit_score",
    by="emi_eligibility",
    ax=axes[1],
    grid=False
)
axes[1].set_title("Credit Score vs EMI Eligibility")
axes[1].set_xlabel("EMI Eligibility")
axes[1].set_ylabel("Credit Score")

# 3. EMI Scenario vs Eligibility
eligibility_by_scenario = pd.crosstab(
    df["emi_scenario"],
    df["emi_eligibility"],
    normalize="index"
) * 100

eligibility_by_scenario.plot(
    kind="bar",
    stacked=True,
    ax=axes[2]
)

axes[2].set_title("EMI Eligibility by Scenario")
axes[2].set_xlabel("EMI Scenario")
axes[2].set_ylabel("Percentage")
axes[2].tick_params(axis="x", rotation=30)

axes[2].legend(
    title="Eligibility",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.suptitle("")
plt.tight_layout()
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_20584\2019539969.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["requested_amount"],
    df["max_monthly_emi"],
    s=10,
    alpha=0.25,
    edgecolors="black",
    linewidths=0.45
)

plt.title("Requested Amount vs Maximum Monthly EMI")
plt.xlabel("Requested Amount")
plt.ylabel("Maximum Monthly EMI")
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_20584\3099034892.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


###Multivariate Analysis

In [19]:
# Focused correlation analysis

correlation_columns = [
    "monthly_salary",
    "years_of_employment",
    "monthly_rent",
    "current_emi_amount",
    "credit_score",
    "bank_balance",
    "emergency_fund",
    "requested_amount",
    "requested_tenure",
    "max_monthly_emi"
]

corr = df[correlation_columns].corr()

fig, ax = plt.subplots(figsize=(11, 8))

im = ax.imshow(
    corr,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1
)

# Add correlation values
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(
            j,
            i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))

ax.set_xticklabels(
    [c.replace("_", " ").title() for c in corr.columns],
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    [c.replace("_", " ").title() for c in corr.columns]
)

ax.set_title(
    "Correlation Matrix of Key Numerical Variables",
    fontsize=14,
    pad=15
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Correlation")

plt.tight_layout()
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_20584\1589740939.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# Pair Plot
import seaborn as sns

pairplot_columns = [
    "age",
    "monthly_salary",
    "years_of_employment",
    "monthly_rent",
    "family_size",
    "dependents",
    "school_fees",
    "college_fees",
    "travel_expenses",
    "groceries_utilities",
    "other_monthly_expenses",
    "current_emi_amount",
    "credit_score",
    "bank_balance",
    "emergency_fund",
    "requested_amount",
    "requested_tenure",
    "max_monthly_emi"
]

# Use a sample so the complete pair plot remains computationally manageable
pairplot_sample = df[pairplot_columns].sample(
    n=5000,
    random_state=42
)

sns.pairplot(
    pairplot_sample,
    diag_kind="hist",
    corner=False,
    plot_kws={
        "alpha": 0.2,
        "s": 8
    }
)

plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_20584\671348095.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# Pair Plot with few essential variables (better visualization)

pairplot_columns = [
    "monthly_salary",
    "credit_score",
    "bank_balance",
    "max_monthly_emi",
    "emi_eligibility"
]

pairplot_data = df[pairplot_columns].sample(
    n=3000,
    random_state=42
)

sns.pairplot(
    pairplot_data,
    vars=pairplot_columns[:-1],
    hue="emi_eligibility",
    diag_kind="hist",
    plot_kws={"alpha": 0.35, "s": 18}
)

plt.suptitle(
    "Multivariate Relationships by EMI Eligibility",
    y=1.02
)

plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_20584\1689387753.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
# Financial Profile by Eligibility

eligibility_profile = (
    df.groupby("emi_eligibility")[
        [
            "monthly_salary",
            "credit_score",
            "bank_balance",
            "emergency_fund",
            "current_emi_amount",
            "requested_amount",
            "max_monthly_emi"
        ]
    ]
    .mean()
    .round(2)
)

eligibility_profile

,monthly_salary,credit_score,bank_balance,emergency_fund,current_emi_amount,requested_amount,max_monthly_emi
emi_eligibility,,,,,,,
Eligible,"78,860.94",725.62,"337,758.97","135,511.70","2,405.85","155,361.20","15,911.96"
High_Risk,"70,197.67",716.16,"293,116.65","118,226.32","3,088.11","261,358.25","11,864.75"
Not_Eligible,"54,257.63",694.11,"215,280.89","86,180.14","5,133.36","428,250.65","4,301.70"


### Key Findings

1. **Class imbalance:** The dataset is dominated by `Not_Eligible` applicants
   (77.29%), while `High_Risk` represents only 4.32%. This makes class-aware
   evaluation important for the classification task.

2. **Financial capacity:** `Eligible` applicants have the highest average
   monthly salary, bank balance, and emergency fund among the three groups.

3. **Existing EMI burden:** `Not_Eligible` applicants have a substantially
   higher average current EMI amount, suggesting greater existing financial
   burden.

4. **Credit profile:** The `Eligible` group has the highest average credit
   score, while `Not_Eligible` has the lowest.

5. **Loan demand:** `Not_Eligible` applicants request substantially larger
   amounts on average than `Eligible` applicants.

6. **Maximum affordable EMI:** `Eligible` applicants have a much higher average
   maximum monthly EMI, while `Not_Eligible` applicants have the lowest.

7. **Scenario differences:** EMI eligibility proportions vary across the five
   EMI scenarios, indicating that the requested EMI purpose may contain useful
   predictive information.

8. **Feature relationships:** The correlation analysis shows meaningful
   relationships among several financial variables, particularly salary,
   bank balance, emergency fund, and maximum monthly EMI.

These findings will be used to guide feature engineering and model development.
They represent associations observed in the dataset and should not be
interpreted as causal relationships.

#**Feature Engineering**

In [23]:
# Create engineered financial and risk features
df_fe = df.copy()

# 1. Total monthly expenses
expense_columns = [
    "monthly_rent",
    "school_fees",
    "college_fees",
    "travel_expenses",
    "groceries_utilities",
    "other_monthly_expenses"
]
df_fe["total_monthly_expenses"] = df_fe[expense_columns].sum(axis=1)


# 2. Financial burden ratios
income = df_fe["monthly_salary"].replace(0, np.nan)

df_fe["debt_to_income"] = (
    df_fe["current_emi_amount"] / income
)
df_fe["expense_to_income"] = (
    df_fe["total_monthly_expenses"] / income
)


# 3. Financial capacity
df_fe["income_after_expenses"] = (
    df_fe["monthly_salary"]
    - df_fe["total_monthly_expenses"]
    - df_fe["current_emi_amount"]
)
df_fe["financial_cushion"] = (
    df_fe["bank_balance"] + df_fe["emergency_fund"]
)
'''df_fe["affordability_ratio"] = (
    df_fe["max_monthly_emi"] / income
)'''


# 4. Loan-request burden
df_fe["requested_payment_burden"] = (
    df_fe["requested_amount"] /
    df_fe["requested_tenure"].replace(0, np.nan)
)
df_fe["loan_to_income"] = (
    df_fe["requested_amount"] / income
)

# 5. Risk / stability indicators
'''df_fe["credit_risk_score"] = (
    850 - df_fe["credit_score"]
)'''
df_fe["employment_stability"] = (
    df_fe["years_of_employment"] /
    df_fe["age"].replace(0, np.nan)
)

# 6. Interaction features
df_fe["salary_credit_interaction"] = (
    df_fe["monthly_salary"] *
    df_fe["credit_score"]
)

df_fe["salary_employment_interaction"] = (
    df_fe["monthly_salary"] *
    df_fe["years_of_employment"]
)

df_fe["balance_emergency_interaction"] = (
    df_fe["bank_balance"] *
    df_fe["emergency_fund"]
)

print(f"Original features : {df.shape[1]}")
print(f"Engineered dataset: {df_fe.shape[1]}")
print(f"New features      : {df_fe.shape[1] - df.shape[1]}")

Original features : 27
Engineered dataset: 38
New features      : 11


In [24]:
# Validate engineered features

engineered_features = [
    "total_monthly_expenses",
    "debt_to_income",
    "expense_to_income",
    "income_after_expenses",
    "financial_cushion",
    "requested_payment_burden",
    "loan_to_income",
    "employment_stability",
    "salary_credit_interaction",
    "salary_employment_interaction",
    "balance_emergency_interaction"
]

validation = pd.DataFrame({
    "Missing": df_fe[engineered_features].isna().sum(),
    "Infinite": np.isinf(df_fe[engineered_features]).sum(),
    "Min": df_fe[engineered_features].min(),
    "Max": df_fe[engineered_features].max()
})

validation

,Missing,Infinite,Min,Max
total_monthly_expenses,0,0,"3,400.00","207,500.00"
debt_to_income,0,0,0.00,7.70
expense_to_income,0,0,0.01,26.78
income_after_expenses,0,0,"-161,883.00","485,540.00"
financial_cushion,0,0,"8,100.00","2,453,600.00"
requested_payment_burden,0,0,416.67,"124,583.33"
loan_to_income,0,0,0.02,294.56
employment_stability,0,0,0.01,0.63
salary_credit_interaction,0,0,0.00,"595,365,600.00"
salary_employment_interaction,0,0,"1,983.50","17,916,084.00"


In [25]:
# Define prediction targets and final model features

classification_target = "emi_eligibility"
regression_target = "max_monthly_emi"

feature_columns = [
    col for col in df_fe.columns
    if col not in [classification_target, regression_target]
]

print(f"Total model input features: {len(feature_columns)}")
print(f"Classification target      : {classification_target}")
print(f"Regression target          : {regression_target}")

Total model input features: 36
Classification target      : emi_eligibility
Regression target          : max_monthly_emi


In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify feature types
categorical_features = df_fe[feature_columns].select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = [
    col for col in feature_columns
    if col not in categorical_features
]

print(f"Numerical features   : {len(numerical_features)}")
print(f"Categorical features : {len(categorical_features)}")

print("\nCategorical columns:")
print(categorical_features)

print("\nNumerical columns:")
print(numerical_features)

Numerical features   : 28
Categorical features : 8

Categorical columns:
['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'emi_scenario']

Numerical columns:
['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure', 'total_monthly_expenses', 'debt_to_income', 'expense_to_income', 'income_after_expenses', 'financial_cushion', 'requested_payment_burden', 'loan_to_income', 'employment_stability', 'salary_credit_interaction', 'salary_employment_interaction', 'balance_emergency_interaction']


In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="drop"
)

print("Preprocessing pipeline configured successfully.")

Preprocessing pipeline configured successfully.


In [28]:
# Prepare feature and target datasets

X_train_fe = X_train.copy()
X_val_fe = X_val.copy()
X_test_fe = X_test.copy()

'''y_class_train = y_class_train.copy()
y_class_val = y_class_val.copy()
y_class_test = y_class_test.copy()

y_reg_train = y_reg_train.copy()
y_reg_val = y_reg_val.copy()
y_reg_test = y_reg_test.copy()
'''
# Add engineered features to each split
engineered_only = [
    col for col in df_fe.columns
    if col not in df.columns
]

X_train_fe = pd.concat(
    [X_train_fe, df_fe.loc[X_train.index, engineered_only]],
    axis=1
)

X_val_fe = pd.concat(
    [X_val_fe, df_fe.loc[X_val.index, engineered_only]],
    axis=1
)

X_test_fe = pd.concat(
    [X_test_fe, df_fe.loc[X_test.index, engineered_only]],
    axis=1
)

'''# Remove target-derived / unsuitable features if present
X_train_fe = X_train_fe.drop(
    columns=["affordability_ratio", "credit_risk_score"],
    errors="ignore")

X_val_fe = X_val_fe.drop(
    columns=["affordability_ratio", "credit_risk_score"],
    errors="ignore")

X_test_fe = X_test_fe.drop(
    columns=["affordability_ratio", "credit_risk_score"],
    errors="ignore")'''

print("Feature datasets prepared successfully.")
print(f"Training features   : {X_train_fe.shape}")
print(f"Validation features : {X_val_fe.shape}")
print(f"Test features       : {X_test_fe.shape}")

Feature datasets prepared successfully.
Training features   : (283360, 36)
Validation features : (60720, 36)
Test features       : (60720, 36)


In [29]:
# Fit preprocessing only on training data

preprocessor.fit(X_train_fe)

X_train_processed = preprocessor.transform(X_train_fe)
X_val_processed = preprocessor.transform(X_val_fe)
X_test_processed = preprocessor.transform(X_test_fe)

print("Preprocessing completed.")
print("-" * 40)
print(f"Training processed   : {X_train_processed.shape}")
print(f"Validation processed : {X_val_processed.shape}")
print(f"Test processed       : {X_test_processed.shape}")

Preprocessing completed.
----------------------------------------
Training processed   : (283360, 54)
Validation processed : (60720, 54)
Test processed       : (60720, 54)


In [30]:
# Final feature-engineering validation

print("FEATURE ENGINEERING VALIDATION")
print("=" * 50)

print(f"Training shape   : {X_train_processed.shape}")
print(f"Validation shape : {X_val_processed.shape}")
print(f"Test shape       : {X_test_processed.shape}")

print("\nFeature dimension consistency:")
print(
    "PASS" if (
        X_train_processed.shape[1]
        == X_val_processed.shape[1]
        == X_test_processed.shape[1]
    )
    else "FAIL"
)

print("\nMissing values:")
print(f"Training   : {np.isnan(X_train_processed).sum():,}")
print(f"Validation : {np.isnan(X_val_processed).sum():,}")
print(f"Test       : {np.isnan(X_test_processed).sum():,}")

print("\nInfinite values:")
print(f"Training   : {np.isinf(X_train_processed).sum():,}")
print(f"Validation : {np.isinf(X_val_processed).sum():,}")
print(f"Test       : {np.isinf(X_test_processed).sum():,}")

FEATURE ENGINEERING VALIDATION
Training shape   : (283360, 54)
Validation shape : (60720, 54)
Test shape       : (60720, 54)

Feature dimension consistency:
PASS

Missing values:
Training   : 0
Validation : 0
Test       : 0

Infinite values:
Training   : 0
Validation : 0
Test       : 0


#**ML Model Development**

In [31]:

# TARGET PREPARATION
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

# Classification target
classification_target = "emi_eligibility"

# Encode the 3 eligibility classes numerically
label_encoder = LabelEncoder()

y_class_train_encoded = label_encoder.fit_transform(y_class_train)
y_class_val_encoded = label_encoder.transform(y_class_val)
y_class_test_encoded = label_encoder.transform(y_class_test)

print("Classification target:", classification_target)
print("Class mapping:")

for encoded, original in enumerate(label_encoder.classes_):
    print(f"  {encoded} -> {original}")


# Regression target
regression_target = "max_monthly_emi"
print("\nRegression target:", regression_target)


# Target validation
print("\nTARGET VALIDATION")
print("=" * 60)

print("\nClassification target distribution:")

print(
    pd.Series(
        y_class_train,
        name=classification_target
    ).value_counts()
)


print("\nRegression target statistics:")

print(
    pd.Series(
        y_reg_train,
        name=regression_target
    ).describe()
)


print("\nTarget sizes:")
print(f"Classification training : {len(y_class_train_encoded):,}")
print(f"Classification validation : {len(y_class_val_encoded):,}")
print(f"Classification test       : {len(y_class_test_encoded):,}")

print(f"\nRegression training      : {len(y_reg_train):,}")
print(f"Regression validation    : {len(y_reg_val):,}")
print(f"Regression test          : {len(y_reg_test):,}")

Classification target: emi_eligibility
Class mapping:
  0 -> Eligible
  1 -> High_Risk
  2 -> Not_Eligible

Regression target: max_monthly_emi

TARGET VALIDATION

Classification target distribution:
emi_eligibility
Not_Eligible    219007
Eligible         52111
High_Risk        12242
Name: count, dtype: int64

Regression target statistics:
count   283,360.00
mean      6,758.54
std       7,732.86
min         500.00
25%         500.00
50%       4,209.60
75%       9,794.40
max      89,166.00
Name: max_monthly_emi, dtype: float64

Target sizes:
Classification training : 283,360
Classification validation : 60,720
Classification test       : 60,720

Regression training      : 283,360
Regression validation    : 60,720
Regression test          : 60,720


###Utility Functions

In [32]:
# CLASSIFICATION EVALUATION UTILITY
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

def evaluate_classification(model, model_name):

    model.fit(
        X_train_processed,
        y_class_train_encoded
    )

    y_pred = model.predict(X_val_processed)
    y_prob = model.predict_proba(X_val_processed)

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_class_val_encoded,
            y_pred
        ),
        "Precision": precision_score(
            y_class_val_encoded,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "Recall": recall_score(
            y_class_val_encoded,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "F1-Score": f1_score(
            y_class_val_encoded,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_class_val_encoded,
            y_prob,
            multi_class="ovr",
            average="weighted"
        )
    }

    return model, results

In [33]:
# REGRESSION EVALUATION UTILITY

#Train and evaluate a regression model on the validation set.
def evaluate_regression(model, model_name):

    model.fit(X_train_processed,y_reg_train)
    y_pred = model.predict(X_val_processed)

    rmse = np.sqrt(mean_squared_error(y_reg_val,y_pred))
    mape = np.mean(np.abs((y_reg_val - y_pred) /np.where(y_reg_val != 0, y_reg_val, np.nan))) * 100

    results = {
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mean_absolute_error(
            y_reg_val,
            y_pred
        ),
        "RÂ²": r2_score(
            y_reg_val,
            y_pred
        ),
        "MAPE": mape
    }

    return model, results

In [34]:
# CLASSIFICATION MODELS

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


classification_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )
}


# Train and evaluate
classification_results = []
trained_classification_models = {}

for model_name, model in classification_models.items():

    print(f"Training {model_name}...")
    trained_model, results = evaluate_classification(
        model,
        model_name
    )

    trained_classification_models[model_name] = trained_model
    classification_results.append(results)


# Comparison table
classification_results_df = (
    pd.DataFrame(classification_results)
    .sort_values("F1-Score", ascending=False)
    .reset_index(drop=True)
)

print("\nClassification Model Comparison")
print("=" * 70)

display(
    classification_results_df.round(4)
)

Training Logistic Regression...


Training Random Forest...


Training XGBoost...



Classification Model Comparison


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,XGBoost,0.97,0.97,0.97,0.97,1.00
1,Random Forest,0.95,0.94,0.95,0.93,1.00
2,Logistic Regression,0.91,0.88,0.91,0.89,0.97


In [35]:
# ============================================================
# 4.4 â€” REGRESSION MODELS
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

regression_models = {

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=2
    ),

    "XGBoost": XGBRegressor(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=2
    )
}


# ============================================================
# TRAIN AND EVALUATE
# ============================================================

regression_results = []
trained_regression_models = {}

for model_name, model in regression_models.items():

    print(f"Training {model_name}...")

    trained_model, results = evaluate_regression(
        model,
        model_name
    )

    trained_regression_models[model_name] = trained_model
    regression_results.append(results)

    print(
        f"Completed | "
        f"RMSE: {results['RMSE']:.2f} | "
        f"RÂ²: {results['RÂ²']:.4f}"
    )


# ============================================================
# COMPARISON
# ============================================================

regression_results_df = (
    pd.DataFrame(regression_results)
    .sort_values("RMSE", ascending=True)
    .reset_index(drop=True)
)

print("\nRegression Model Comparison")
print("=" * 70)

display(
    regression_results_df.round(4)
)

Training Linear Regression...


Completed | RMSE: 4106.33 | RÂ²: 0.7213
Training Random Forest...


Completed | RMSE: 1205.33 | RÂ²: 0.9760
Training XGBoost...


Completed | RMSE: 885.16 | RÂ²: 0.9870

Regression Model Comparison


,Model,RMSE,MAE,RÂ²,MAPE
0,XGBoost,885.16,415.44,0.99,16.36
1,Random Forest,"1,205.33",590.49,0.98,18.02
2,Linear Regression,"4,106.33","2,934.78",0.72,188.78


In [36]:
# MODEL COMPARISON & SELECTION
print("CLASSIFICATION MODEL COMPARISON")
print("=" * 70)

display(
    classification_results_df.round(4)
)

print("\nREGRESSION MODEL COMPARISON")
print("=" * 70)

display(
    regression_results_df.round(4)
)

# Best validation models
best_classification_name = (
    classification_results_df
    .sort_values("F1-Score", ascending=False)
    .iloc[0]["Model"]
)

best_regression_name = (
    regression_results_df
    .sort_values("RMSE", ascending=True)
    .iloc[0]["Model"]
)

print("\nBEST VALIDATION MODELS")
print("=" * 70)

print(f"Classification : {best_classification_name}")
print(f"Regression     : {best_regression_name}")

CLASSIFICATION MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,XGBoost,0.97,0.97,0.97,0.97,1.00
1,Random Forest,0.95,0.94,0.95,0.93,1.00
2,Logistic Regression,0.91,0.88,0.91,0.89,0.97



REGRESSION MODEL COMPARISON


,Model,RMSE,MAE,RÂ²,MAPE
0,XGBoost,885.16,415.44,0.99,16.36
1,Random Forest,"1,205.33",590.49,0.98,18.02
2,Linear Regression,"4,106.33","2,934.78",0.72,188.78



BEST VALIDATION MODELS
Classification : XGBoost
Regression     : XGBoost


In [37]:
# CLASSIFICATION PER-CLASS PERFORMANCE
from sklearn.metrics import classification_report

best_classifier = trained_classification_models[
    best_classification_name
]

y_best_class_pred = best_classifier.predict(
    X_val_processed
)

print(
    classification_report(
        y_class_val_encoded,
        y_best_class_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

    Eligible     0.9478    0.9764    0.9618     11166
   High_Risk     0.8013    0.5303    0.6382      2623
Not_Eligible     0.9837    0.9952    0.9894     46931

    accuracy                         0.9717     60720
   macro avg     0.9109    0.8340    0.8632     60720
weighted avg     0.9692    0.9717    0.9692     60720



In [38]:
# FINAL TEST EVALUATION
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Selected models
final_classification_model = trained_classification_models[
    best_classification_name
]
final_regression_model = trained_regression_models[
    best_regression_name
]


# CLASSIFICATION â€” TEST SET
y_test_class_pred = final_classification_model.predict(
    X_test_processed
)
y_test_class_prob = final_classification_model.predict_proba(
    X_test_processed
)

classification_test_results = {
    "Model": best_classification_name,
    "Accuracy": accuracy_score(
        y_class_test_encoded,
        y_test_class_pred
    ),
    "Precision": precision_score(
        y_class_test_encoded,
        y_test_class_pred,
        average="weighted",
        zero_division=0
    ),
    "Recall": recall_score(
        y_class_test_encoded,
        y_test_class_pred,
        average="weighted",
        zero_division=0
    ),
    "F1-Score": f1_score(
        y_class_test_encoded,
        y_test_class_pred,
        average="weighted",
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_class_test_encoded,
        y_test_class_prob,
        multi_class="ovr",
        average="weighted"
    )
}


# REGRESSION â€” TEST SET
y_test_reg_pred = final_regression_model.predict(
    X_test_processed
)

regression_test_results = {
    "Model": best_regression_name,
    "RMSE": np.sqrt(
        mean_squared_error(
            y_reg_test,
            y_test_reg_pred
        )
    ),
    "MAE": mean_absolute_error(
        y_reg_test,
        y_test_reg_pred
    ),
    "RÂ²": r2_score(
        y_reg_test,
        y_test_reg_pred
    ),
    "MAPE": np.mean(
        np.abs(
            (y_reg_test - y_test_reg_pred) /
            np.where(
                y_reg_test != 0,
                y_reg_test,
                np.nan
            )
        )
    ) * 100
}


# FINAL RESULTS
print("FINAL TEST RESULTS")
print("=" * 70)

print("\nClassification:")
display(
    pd.DataFrame([classification_test_results]).round(4)
)
print("\nRegression:")
display(
    pd.DataFrame([regression_test_results]).round(4)
)


# CLASSIFICATION PER-CLASS PERFORMANCE
print("\nClassification Report:")
print(
    classification_report(
        y_class_test_encoded,
        y_test_class_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

FINAL TEST RESULTS

Classification:


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,XGBoost,0.97,0.97,0.97,0.97,1.00



Regression:


,Model,RMSE,MAE,RÂ²,MAPE
0,XGBoost,878.47,415.67,0.99,16.18



Classification Report:
              precision    recall  f1-score   support

    Eligible     0.9495    0.9761    0.9626     11167
   High_Risk     0.8092    0.5273    0.6385      2623
Not_Eligible     0.9829    0.9955    0.9892     46930

    accuracy                         0.9717     60720
   macro avg     0.9139    0.8330    0.8634     60720
weighted avg     0.9693    0.9717    0.9691     60720



###Class Imbalance Handling

In [39]:
# CLASS WEIGHTS
from sklearn.utils.class_weight import compute_sample_weight

# Calculate weights from training labels only
class_sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_class_train_encoded
)

print("Class distribution:")
print(
    pd.Series(
        y_class_train_encoded
    ).value_counts().sort_index()
)

print("\nClass weights:")
for class_id in np.unique(y_class_train_encoded):
    weight = class_sample_weights[
        y_class_train_encoded == class_id
    ][0]

    print(
        f"{class_id} -> "
        f"{label_encoder.classes_[class_id]}: "
        f"{weight:.2f}"
    )

Class distribution:
0     52111
1     12242
2    219007
Name: count, dtype: int64

Class weights:
0 -> Eligible: 1.81
1 -> High_Risk: 7.72
2 -> Not_Eligible: 0.43


In [40]:
# WEIGHTED XGBOOST
weighted_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

print("Training weighted XGBoost...")

weighted_xgb.fit(
    X_train_processed,
    y_class_train_encoded,
    sample_weight=class_sample_weights
)

print("Weighted XGBoost training completed.")


# Validation evaluation
y_weighted_pred = weighted_xgb.predict(
    X_val_processed
)
y_weighted_prob = weighted_xgb.predict_proba(
    X_val_processed
)

weighted_results = {
    "Model": "Weighted XGBoost",
    "Accuracy": accuracy_score(
        y_class_val_encoded,
        y_weighted_pred
    ),
    "Precision": precision_score(
        y_class_val_encoded,
        y_weighted_pred,
        average="weighted",
        zero_division=0
    ),
    "Recall": recall_score(
        y_class_val_encoded,
        y_weighted_pred,
        average="weighted",
        zero_division=0
    ),
    "F1-Score": f1_score(
        y_class_val_encoded,
        y_weighted_pred,
        average="weighted",
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_class_val_encoded,
        y_weighted_prob,
        multi_class="ovr",
        average="weighted"
    )
}

print("\nWeighted XGBoost â€” Validation Results")
print("=" * 60)

display(
    pd.DataFrame([weighted_results]).round(4)
)

Training weighted XGBoost...


Weighted XGBoost training completed.



Weighted XGBoost â€” Validation Results


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Weighted XGBoost,0.94,0.97,0.94,0.95,1.00


In [41]:
# HIGH_RISK PERFORMANCE â€” WEIGHTED XGBOOST

print(
    classification_report(
        y_class_val_encoded,
        y_weighted_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

    Eligible     0.9814    0.9060    0.9422     11166
   High_Risk     0.4025    0.9409    0.5639      2623
Not_Eligible     0.9979    0.9416    0.9689     46931

    accuracy                         0.9350     60720
   macro avg     0.7939    0.9295    0.8250     60720
weighted avg     0.9692    0.9350    0.9465     60720



### Class Imbalance Experiment â€” Conclusion

Class weighting substantially improved High_Risk recall from approximately
48% to 90%. However, this improvement came with a reduction in overall
accuracy and weighted F1-score, while High_Risk precision decreased
substantially.

Therefore, the original XGBoost model was retained as the final
classification model because it provided stronger overall validation
performance. The weighted XGBoost model was not selected for deployment.

#**MLflow Experiment Tracking & Model Registry**

In [42]:
# MLFLOW SETUP
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import os

print("MLflow version:", mlflow.__version__)

# Experiment names
CLASSIFICATION_EXPERIMENT = "EMIPredict_Classification"
REGRESSION_EXPERIMENT = "EMIPredict_Regression"

# Allow filesystem tracking backend
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Local MLflow tracking
mlflow.set_tracking_uri("file:./mlruns")

print("Tracking URI:", mlflow.get_tracking_uri())

# Create or retrieve experiments
classification_experiment = mlflow.set_experiment(
    CLASSIFICATION_EXPERIMENT
)

regression_experiment = mlflow.set_experiment(
    REGRESSION_EXPERIMENT
)

print("\nMLflow experiments ready.")
print("Classification:", classification_experiment.name)
print("Regression    :", regression_experiment.name)

2026/08/23 19:57:50 INFO mlflow.tracking.fluent: Experiment with name 'EMIPredict_Classification' does not exist. Creating a new experiment.


2026/08/23 19:57:50 INFO mlflow.tracking.fluent: Experiment with name 'EMIPredict_Regression' does not exist. Creating a new experiment.


MLflow version: 3.15.1
Tracking URI: file:./mlruns

MLflow experiments ready.
Classification: EMIPredict_Classification
Regression    : EMIPredict_Regression


In [43]:
# ============================================================
# 5.2 â€” MLFLOW LOGGING UTILITY
# ============================================================

from xgboost import XGBClassifier, XGBRegressor
from sklearn.base import BaseEstimator

def log_mlflow_run(
    experiment_name,
    run_name,
    model,
    model_type,
    metrics,
    params=None
):
    """
    Log a trained model to MLflow with parameters, metrics, and artifact.
    """

    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=run_name) as run:

        # Log model parameters
        if params:
            mlflow.log_params(params)

        # Log evaluation metrics
        mlflow.log_metrics({
            key: float(value)
            for key, value in metrics.items()
            if value is not None
        })

        # Log model artifact based on its type
        if isinstance(model, (XGBClassifier, XGBRegressor)):
            mlflow.xgboost.log_model(
                model,
                name="model"
            )
        elif isinstance(model, BaseEstimator):
            mlflow.sklearn.log_model(
                model,
                name="model"
            )
        else:
            print(f"Warning: Model type {type(model)} not supported for logging.")

        print(
            f"Logged: {run_name} | "
            f"Run ID: {run.info.run_id}"
        )

        return run.info.run_id

In [44]:
# ============================================================
# 5.3 â€” CLASSIFICATION MLFLOW TRACKING
# ============================================================

for model_name, model in trained_classification_models.items():

    # Get validation metrics for this model
    model_metrics = (
        classification_results_df[
            classification_results_df["Model"] == model_name
        ]
        .iloc[0]
        .drop("Model")
        .to_dict()
    )

    # Get model parameters
    model_params = model.get_params()

    log_mlflow_run(
        experiment_name=CLASSIFICATION_EXPERIMENT,
        run_name=model_name,
        model=model,
        model_type="classification",
        metrics=model_metrics,
        params=model_params
    )

print("\nAll classification models logged to MLflow.")

Logged: Logistic Regression | Run ID: 8642225267cf4efaa0a6ca7baaa06ba0


Logged: Random Forest | Run ID: ef10989a2d3249ac934b38c86f6bccb0


Logged: XGBoost | Run ID: dc814f22b9f94ed4820a0c255331a1a8

All classification models logged to MLflow.


In [45]:
# ============================================================
# 5.4 â€” REGRESSION MLFLOW TRACKING
# ============================================================

for model_name, model in trained_regression_models.items():

    # Get validation metrics for this model
    model_metrics = (
        regression_results_df[
            regression_results_df["Model"] == model_name
        ]
        .iloc[0]
        .drop("Model")
        .to_dict()
    )

    # Get model parameters
    model_params = model.get_params()

    log_mlflow_run(
        experiment_name=REGRESSION_EXPERIMENT,
        run_name=model_name,
        model=model,
        model_type="regression",
        metrics=model_metrics,
        params=model_params
    )

print("\nAll regression models logged to MLflow.")

Logged: Linear Regression | Run ID: d923d376cbe14cf388e243d8739ea4aa


Logged: Random Forest | Run ID: 62e6705a43b64ce9ab7d1556070c6016


Logged: XGBoost | Run ID: 664231fb149c433b959df4c72e714456

All regression models logged to MLflow.


In [46]:
# ============================================================
# 5.5 â€” MLFLOW EXPERIMENT VERIFICATION
# ============================================================

def get_mlflow_runs(experiment_name):
    experiment = mlflow.get_experiment_by_name(
        experiment_name
    )

    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )

    return runs


classification_runs = get_mlflow_runs(
    CLASSIFICATION_EXPERIMENT
)

regression_runs = get_mlflow_runs(
    REGRESSION_EXPERIMENT
)


print("Classification MLflow Runs")
print("=" * 70)

display(
    classification_runs[
        [
            "run_id",
            "tags.mlflow.runName",
            "metrics.Accuracy",
            "metrics.Precision",
            "metrics.Recall",
            "metrics.F1-Score",
            "metrics.ROC-AUC"
        ]
    ].round(4)
)


print("\nRegression MLflow Runs")
print("=" * 70)

display(
    regression_runs[
        [
            "run_id",
            "tags.mlflow.runName",
            "metrics.RMSE",
            "metrics.MAE",
            "metrics.RÂ²",
            "metrics.MAPE"
        ]
    ].round(4)
)

Classification MLflow Runs


,run_id,tags.mlflow.runName,metrics.Accuracy,metrics.Precision,metrics.Recall,metrics.F1-Score,metrics.ROC-AUC
0,dc814f22b9f94ed4820a0c255331a1a8,XGBoost,0.97,0.97,0.97,0.97,1.00
1,ef10989a2d3249ac934b38c86f6bccb0,Random Forest,0.95,0.94,0.95,0.93,1.00
2,8642225267cf4efaa0a6ca7baaa06ba0,Logistic Regression,0.91,0.88,0.91,0.89,0.97



Regression MLflow Runs


,run_id,tags.mlflow.runName,metrics.RMSE,metrics.MAE,metrics.RÂ²,metrics.MAPE
0,664231fb149c433b959df4c72e714456,XGBoost,885.16,415.44,0.99,16.36
1,62e6705a43b64ce9ab7d1556070c6016,Random Forest,"1,205.33",590.49,0.98,18.02
2,d923d376cbe14cf388e243d8739ea4aa,Linear Regression,"4,106.33","2,934.78",0.72,188.78


In [47]:
# ============================================================
# 5.5.1 â€” VERIFY PHASE 4 RESULTS
# ============================================================

print("Current Phase 4 Regression Results")
print("=" * 70)

display(
    regression_results_df.round(4)
)

print("\nCurrent Phase 4 Classification Results")
print("=" * 70)

display(
    classification_results_df.round(4)
)

Current Phase 4 Regression Results


,Model,RMSE,MAE,RÂ²,MAPE
0,XGBoost,885.16,415.44,0.99,16.36
1,Random Forest,"1,205.33",590.49,0.98,18.02
2,Linear Regression,"4,106.33","2,934.78",0.72,188.78



Current Phase 4 Classification Results


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,XGBoost,0.97,0.97,0.97,0.97,1.00
1,Random Forest,0.95,0.94,0.95,0.93,1.00
2,Logistic Regression,0.91,0.88,0.91,0.89,0.97


In [48]:
from mlflow import MlflowClient

client = MlflowClient()

CLASSIFICATION_MODEL_NAME = "EMIPredict_Classification_XGBoost"
REGRESSION_MODEL_NAME = "EMIPredict_Regression_XGBoost"


# ------------------------------------------------------------
# Find the latest XGBoost classification run
# ------------------------------------------------------------

classification_runs = mlflow.search_runs(
    experiment_names=[CLASSIFICATION_EXPERIMENT],
    filter_string="tags.mlflow.runName = 'XGBoost'",
    order_by=["metrics.`F1-Score` DESC"],
    max_results=1
)

classification_run_id = classification_runs.iloc[0]["run_id"]


# ------------------------------------------------------------
# Find the latest XGBoost regression run
# ------------------------------------------------------------

regression_runs = mlflow.search_runs(
    experiment_names=[REGRESSION_EXPERIMENT],
    filter_string="tags.mlflow.runName = 'XGBoost'",
    order_by=["metrics.RMSE ASC"],
    max_results=1
)

regression_run_id = regression_runs.iloc[0]["run_id"]


# ------------------------------------------------------------
# Register models
# ------------------------------------------------------------

classification_model_uri = (
    f"runs:/{classification_run_id}/model"
)

regression_model_uri = (
    f"runs:/{regression_run_id}/model"
)

classification_registration = mlflow.register_model(
    model_uri=classification_model_uri,
    name=CLASSIFICATION_MODEL_NAME
)

regression_registration = mlflow.register_model(
    model_uri=regression_model_uri,
    name=REGRESSION_MODEL_NAME
)


print("Model Registry")
print("=" * 70)

print(
    f"Classification: {CLASSIFICATION_MODEL_NAME}"
)
print(
    f"Version: {classification_registration.version}"
)

print()

print(
    f"Regression: {REGRESSION_MODEL_NAME}"
)
print(
    f"Version: {regression_registration.version}"
)

Successfully registered model 'EMIPredict_Classification_XGBoost'.
2026/08/23 19:59:16 WARNING mlflow.tracking._model_registry.fluent: Run with id dc814f22b9f94ed4820a0c255331a1a8 has no artifacts at artifact path 'model', registering model based on models:/m-9a70edcf05024be4b815a6a2765b8d99 instead


Created version '1' of model 'EMIPredict_Classification_XGBoost'.
Successfully registered model 'EMIPredict_Regression_XGBoost'.
2026/08/23 19:59:16 WARNING mlflow.tracking._model_registry.fluent: Run with id 664231fb149c433b959df4c72e714456 has no artifacts at artifact path 'model', registering model based on models:/m-1f68ae6af37b46f9bad65c587d2a95c1 instead


Model Registry
Classification: EMIPredict_Classification_XGBoost
Version: 1

Regression: EMIPredict_Regression_XGBoost
Version: 1


Created version '1' of model 'EMIPredict_Regression_XGBoost'.


In [49]:
# ============================================================
# 5.7 â€” MODEL REGISTRY VERIFICATION
# ============================================================

print("Registered Models")
print("=" * 70)

for model_name in [
    CLASSIFICATION_MODEL_NAME,
    REGRESSION_MODEL_NAME
]:
    versions = client.search_model_versions(
        filter_string=f"name='{model_name}'"
    )

    print(f"\nModel: {model_name}")

    for version in versions:
        print(f"  Version : {version.version}")
        print(f"  Run ID  : {version.run_id}")
        print(f"  Status  : {version.status}")

Registered Models

Model: EMIPredict_Classification_XGBoost
  Version : 1
  Run ID  : dc814f22b9f94ed4820a0c255331a1a8
  Status  : READY

Model: EMIPredict_Regression_XGBoost
  Version : 1
  Run ID  : 664231fb149c433b959df4c72e714456
  Status  : READY


# **Deployment Artifact Export**

Export the selected models, evaluation summaries, and a representative data sample for the Streamlit dashboard.


In [50]:
# Save deployable models and dashboard data
import json
import joblib

ARTIFACT_DIR = Path("../artifacts")
RESULTS_DIR = Path("../results")
DATA_DIR = Path("../data")
for directory in (ARTIFACT_DIR, RESULTS_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

model_bundle = {
    "preprocessor": preprocessor,
    "classifier": final_classification_model,
    "regressor": final_regression_model,
    "label_encoder": label_encoder,
    "feature_columns": feature_columns,
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "engineered_features": engineered_features
}
joblib.dump(model_bundle, ARTIFACT_DIR / "emi_predict_models.joblib")

classification_results_df.to_csv(RESULTS_DIR / "classification_model_comparison.csv", index=False)
regression_results_df.to_csv(RESULTS_DIR / "regression_model_comparison.csv", index=False)
pd.DataFrame([classification_test_results]).to_csv(RESULTS_DIR / "classification_test_metrics.csv", index=False)
pd.DataFrame([regression_test_results]).to_csv(RESULTS_DIR / "regression_test_metrics.csv", index=False)
df.sample(n=min(1000, len(df)), random_state=42).to_csv(DATA_DIR / "dashboard_sample.csv", index=False)

dashboard_summary = {
    "best_classification_model": best_classification_name,
    "best_regression_model": best_regression_name,
    "classification_test_metrics": {key: float(value) if isinstance(value, (np.floating, float, int)) else value for key, value in classification_test_results.items()},
    "regression_test_metrics": {key: float(value) if isinstance(value, (np.floating, float, int)) else value for key, value in regression_test_results.items()}
}
(RESULTS_DIR / "dashboard_summary.json").write_text(json.dumps(dashboard_summary, indent=2), encoding="utf-8")

print("Deployment artifacts saved to:", ARTIFACT_DIR.resolve())
print("Dashboard results saved to:", RESULTS_DIR.resolve())


Deployment artifacts saved to: C:\Users\user\Documents\Codex\2026-08-23\i\outputs\EMIPredict_GitHub\artifacts
Dashboard results saved to: C:\Users\user\Documents\Codex\2026-08-23\i\outputs\EMIPredict_GitHub\results
